# VALIDACION CON DATOS DE LABORATORIO

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("====================================================================")
print("     MÓDULO DE AUDITORÍA CONTINUA: VALIDACIÓN CONTRA LABORATORIO   ")
print("====================================================================\n")

# 1. Carga de Predicciones Continuas del Soft-Sensor (Alta Frecuencia)
df_predictions = pd.read_csv('data/final/soft_sensor_continuous_predictions.csv')
df_predictions['timestamp'] = pd.to_datetime(df_predictions['timestamp'])
df_predictions = df_predictions.sort_values('timestamp')

# 2. Carga de Ensayos Reales de Laboratorio Limpios (Baja Frecuencia)
df_lab = pd.read_csv('data/processed/assays_clean_no_duplicates.csv')
df_lab['timestamp'] = pd.to_datetime(df_lab['timestamp'])
df_lab = df_lab.sort_values('timestamp')

print(f"[DataOps] Registros continuos del Soft-Sensor: {len(df_predictions)}")
print(f"[DataOps] Ensayos analíticos reales de Lab:    {len(df_lab)}")

# 3. Alineamiento Temporal Dinámico (merge_asof)
# Buscamos la predicción del soft-sensor más cercana en el pasado para cada dato de laboratorio
# tolerance=pd.Timedelta('30min') asegura que no crucemos datos con desfases absurdos
df_audit = pd.merge_asof(
    df_lab, 
    df_predictions, 
    on='timestamp', 
    direction='backward', 
    tolerance=pd.Timedelta('30min')
)

# Eliminar filas donde no hubo coincidencia temporal dentro de la tolerancia
df_audit = df_audit.dropna(subset=['pCu_Est', 'pFe_Est', 'pMo_Est']).reset_index(drop=True)
print(f"[Match] Puntos de control síncronos encontrados para auditoría: {len(df_audit)}\n")

# 4. Mapeo de pares analíticos (Verdad Terreno vs Estimación)
elements_mapping = {
    'Cobre (pCu)': {'real': 'pCu', 'pred': 'pCu_Est'},
    'Hierro (pFe)': {'real': 'pFe', 'pred': 'pFe_Est'},
    'Molibdeno (pMo)': {'real': 'pMo', 'pred': 'pMo_Est'}
}

# 5. Cálculo y Despliegue de Métricas Globales de Producción
print("=== CUADRO ESTADÍSTICO DE VALIDACIÓN DEL SOFT-SENSOR CONTINUO ===")
print(f"{'Elemento Mineral':<18} | {'RMSE':<8} | {'MAE':<8} | {'R² Global':<8}")
print("-" * 55)

for name, keys in elements_mapping.items():
    y_real = df_audit[keys['real']].values
    y_pred = df_audit[keys['pred']].values
    
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mae = mean_absolute_error(y_real, y_pred)
    r2 = r2_score(y_real, y_pred)
    
    print(f"{name:<18} | {rmse:.4f} | {mae:.4f} | {r2:.4f}")

# 6. Exportar matriz de auditoría para generar gráficos en la Tesis (Scatter plots / Error histórico)
df_audit.to_csv('data/final/soft_sensor_audit_results.csv', index=False)
print("\n[INFO] Matriz unificada de auditoría exportada en: 'soft_sensor_audit_results.csv'")
print("Recomendación: Usa este archivo para graficar Ley Real vs Ley Estimada a lo largo del tiempo.")

     MÓDULO DE AUDITORÍA CONTINUA: VALIDACIÓN CONTRA LABORATORIO   

[DataOps] Registros continuos del Soft-Sensor: 18124
[DataOps] Ensayos analíticos reales de Lab:    611
[Match] Puntos de control síncronos encontrados para auditoría: 533

=== CUADRO ESTADÍSTICO DE VALIDACIÓN DEL SOFT-SENSOR CONTINUO ===
Elemento Mineral   | RMSE     | MAE      | R² Global
-------------------------------------------------------
Cobre (pCu)        | 2.2974 | 1.7997 | -0.6043
Hierro (pFe)       | 1.8473 | 1.4417 | -1.1824
Molibdeno (pMo)    | 0.8553 | 0.7141 | -0.6434

[INFO] Matriz unificada de auditoría exportada en: 'soft_sensor_audit_results.csv'
Recomendación: Usa este archivo para graficar Ley Real vs Ley Estimada a lo largo del tiempo.


In [5]:
import pandas as pd
import numpy as np

print("====================================================================")
print(" PARTE 1: REMUESTREO RÍGIDO DE PREDICCIONES EN TURNOS DE 12 HORAS ")
print("====================================================================\n")

# 1. Carga de Predicciones Continuas del Soft-Sensor (Alta Frecuencia)
df_predictions = pd.read_csv('data/final/soft_sensor_continuous_predictions.csv')
df_predictions['timestamp'] = pd.to_datetime(df_predictions['timestamp'])

# 2. Configurar el timestamp como índice para poder usar las funciones de tiempo de Pandas
df_predictions = df_predictions.set_index('timestamp').sort_index()

# 3. Aplicar Resample de 12 Horas especificando el origen a las 00:00 horas ('0h')
# '12h' -> Agrupa en bloques de 12 horas
# origin='0h' -> Fuerza a que los bloques inicien exactamente a las 00:00:00 y 12:00:00
# closed='left' -> El intervalo incluye el límite izquierdo (ej: [00:00:00, 12:00:00) )
# label='left' -> Etiqueta el promedio con el inicio de la ventana (00:00 o 12:00)
df_resampled = df_predictions.resample('12h', origin='0h', closed='left', label='left').agg({
    'pCu_Est': 'mean',
    'pFe_Est': 'mean',
    'pMo_Est': 'mean',
    'pZn_Est': 'mean',
    'W0_Standard': 'count'  # Usamos una columna para contar cuántas muestras puntuales entraron en las 12h
})

# Renombrar la columna de conteo para auditoría de calidad de datos
df_resampled = df_resampled.rename(columns={'W0_Standard': 'lecturas_count'})

# 4. Filtrar ventanas vacías o con muy pocas lecturas (Control de calidad)
# Si en 12 horas deberías tener muchas lecturas, filtramos donde el analizador estuvo apagado (menos de 15 lecturas)
min_lecturas_por_turno = 15
df_resampled_clean = df_resampled[df_resampled['lecturas_count'] >= min_lecturas_por_turno]

# 5. Resetear el índice para volver a tener la columna timestamp limpia
df_soft_sensor_12h = df_resampled_clean.reset_index()

# Renombrar columnas para dejar en claro que ya son promedios de turno compuestos
df_soft_sensor_12h = df_soft_sensor_12h.rename(columns={
    'pCu_Est': 'pCu_Est_12h',
    'pFe_Est': 'pFe_Est_12h',
    'pMo_Est': 'pMo_Est_12h',
    'pZn_Est': 'pZn_Est_12h'
})

print(f"[DataOps] Registros originales procesados: {len(df_predictions)}")
print(f"[DataOps] Turnos de 12H válidos generados:  {len(df_soft_sensor_12h)}")

# 6. Exportar el nuevo dataset intermedio
output_filename = 'data/final/soft_sensor_predictions_12h_averages.csv'
df_soft_sensor_12h.to_csv(output_filename, index=False)

print(f"\n[ÉXITO] Archivo generado y guardado como: '{output_filename}'")
print("Revisa este archivo. Verás que los timestamps están perfectamente clavados cada 12 horas (00:00:00 y 12:00:00).")
print("Cuando me confirmes que este CSV se creó correctamente, pasamos a la Parte 2 para cruzarlos con el laboratorio.")

 PARTE 1: REMUESTREO RÍGIDO DE PREDICCIONES EN TURNOS DE 12 HORAS 

[DataOps] Registros originales procesados: 18124
[DataOps] Turnos de 12H válidos generados:  569

[ÉXITO] Archivo generado y guardado como: 'data/final/soft_sensor_predictions_12h_averages.csv'
Revisa este archivo. Verás que los timestamps están perfectamente clavados cada 12 horas (00:00:00 y 12:00:00).
Cuando me confirmes que este CSV se creó correctamente, pasamos a la Parte 2 para cruzarlos con el laboratorio.


In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("====================================================================")
print("     MÓDULO DE AUDITORÍA CONTINUA: CRUCE SÍNCRONO CON LABORATORIO   ")
print("====================================================================\n")

# 1. Carga de los Promedios de Turno del Soft-Sensor (Generados en la Parte 1)
df_predictions_12h = pd.read_csv('data/final/soft_sensor_predictions_12h_averages.csv')
df_predictions_12h['timestamp'] = pd.to_datetime(df_predictions_12h['timestamp'])

# 2. Carga de Ensayos Reales de Laboratorio (Baja Frecuencia)
df_lab = pd.read_csv('data/processed/assays_clean_no_duplicates.csv')
df_lab['timestamp'] = pd.to_datetime(df_lab['timestamp'])

print(f"[DataOps] Turnos promediados del Soft-Sensor (12h): {len(df_predictions_12h)}")
print(f"[DataOps] Ensayos analíticos reales de Lab:        {len(df_lab)}")

# 3. Cruce Síncrono Exacto (Inner Merge)
# Como ambos archivos tienen los timestamps alineados (00:00 y 12:00), un merge simple basta
df_audit = pd.merge(df_lab, df_predictions_12h, on='timestamp', how='inner')

print(f"[Match] Pares analíticos síncronos encontrados para auditoría: {len(df_audit)}\n")

# 4. Mapeo de pares analíticos (Verdad Terreno vs Estimación Compuesta de 12h)
elements_mapping = {
    'Cobre (pCu)': {'real': 'pCu', 'pred': 'pCu_Est_12h'},
    'Hierro (pFe)': {'real': 'pFe', 'pred': 'pFe_Est_12h'},
    'Molibdeno (pMo)': {'real': 'pMo', 'pred': 'pMo_Est_12h'}
}

# 5. Cálculo y Despliegue de Métricas Globales Corregidas
print("=== CUADRO ESTADÍSTICO DE VALIDACIÓN DEL SOFT-SENSOR (PROMEDIOS DE TURNO 12H) ===")
print(f"{'Elemento Mineral':<18} | {'RMSE':<8} | {'MAE':<8} | {'R² Global':<8}")
print("-" * 55)

for name, keys in elements_mapping.items():
    y_real = df_audit[keys['real']].values
    y_pred = df_audit[keys['pred']].values
    
    # Calcular estadísticos analíticos
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mae = mean_absolute_error(y_real, y_pred)
    r2 = r2_score(y_real, y_pred)
    
    print(f"{name:<18} | {rmse:.4f} | {mae:.4f} | {r2:.4f}")

# 6. Exportar matriz consolidada final para tus gráficos de Tesis
df_audit.to_csv('soft_sensor_audit_results.csv', index=False)
print("\n[INFO] Matriz final de auditoría guardada como: 'soft_sensor_audit_results.csv'")
print("[INFO] Listo. Puedes usar este archivo para tus gráficos de dispersión (Scatter Plots).")

     MÓDULO DE AUDITORÍA CONTINUA: CRUCE SÍNCRONO CON LABORATORIO   

[DataOps] Turnos promediados del Soft-Sensor (12h): 569
[DataOps] Ensayos analíticos reales de Lab:        611
[Match] Pares analíticos síncronos encontrados para auditoría: 566

=== CUADRO ESTADÍSTICO DE VALIDACIÓN DEL SOFT-SENSOR (PROMEDIOS DE TURNO 12H) ===
Elemento Mineral   | RMSE     | MAE      | R² Global
-------------------------------------------------------
Cobre (pCu)        | 2.3114 | 1.7605 | -0.4099
Hierro (pFe)       | 1.8140 | 1.4002 | -0.7933
Molibdeno (pMo)    | 0.8916 | 0.7369 | -0.4407

[INFO] Matriz final de auditoría guardada como: 'soft_sensor_audit_results.csv'
[INFO] Listo. Puedes usar este archivo para tus gráficos de dispersión (Scatter Plots).
